In [1]:
#the import packages
import requests
import pandas as pd
from pandas import json_normalize
import requests
import os
from pathlib import Path
from datetime import datetime, timezone,timedelta,time
from scipy import stats
import json
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import sklearn as sklearn

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
userInputData = loadDataFromFile("UserPrevious experiments-preprocessed")
timeSeriesData_BIG = loadDataFromFile("Data:Previous experiments-preprocessed")

NameError: name 'loadDataFromFile' is not defined

In [ ]:
def loadDataFromFile(file_name):
    script_dir = Path().resolve().parent

    data_folder = script_dir / 'dataAnalysis and machine learning'/'data'
    print(data_folder)
    data_folder.mkdir(exist_ok=True)
    
    file_path = data_folder / (file_name + ".json")
    
    if file_path.exists():
        df = pd.read_json(file_path)
        print(f"Loaded {len(df)} records from {file_path}")
        return df
    else:
        print(f"File {file_path} does not exist.")
        return None    

In [ ]:
userInputData.head(20)

In [ ]:
# Convert back to timedelta
timeSeriesData_BIG['timestamp'] = pd.to_timedelta(timeSeriesData_BIG['timestamp'], unit='ms')

# Convert back to datetime

timeSeriesData_BIG ["datetime_timestamp"]= timeSeriesData_BIG['datetime_timestamp'].transform(
    lambda x: pd.to_datetime(x, unit='ms')
)
columns_datetime= [
       'date of experiment', 'actual timestamp StartingExperiment',
       'actual timestamp EndingExperiment', ]
columns_timedelta = ['time taken total',
       'timestamp InsertingSource timedelta',
       'time taken after insertion']

userInputData.loc[:,columns_datetime] = userInputData.loc[:,columns_datetime].apply(lambda x:pd.to_datetime(x, unit='ms'))
userInputData.loc[:,columns_timedelta] = userInputData.loc[:,columns_timedelta].apply(lambda x:pd.to_timedelta(x, unit='ms'))

In [ ]:
timeSeriesData_BIG

In [ ]:
timeSeriesData_BIG.columns

In [ ]:
#timeSeriesData_BIG = timeSeriesData_BIG.drop(['VOC standard scaled', 'VOC gradient'],axis=1)

In [ ]:
#timeSeriesData_BIG = timeSeriesData_BIG.set_index("seconds")

In [ ]:
timeSeriesData_BIG

In [ ]:
# Split back into dict
dict_of_timeseries = {k: v.drop(columns="keys").reset_index(drop=True) 
             for k, v in timeSeriesData_BIG.groupby("keys")}

In [ ]:
#create rolling windows 
window_ranges = [5,15,30,60,120]
for window_size in window_ranges:
    
    column_name = "VOC window " + str(window_size)
    timeSeriesData_BIG[column_name] = np.nan
    for index, row in userInputData.iterrows():
        outer_mask = timeSeriesData_BIG["keys"] == index
        particular_experiment =  timeSeriesData_BIG.loc[outer_mask].copy()
        
        sensors = particular_experiment["sensors"].unique()
    
        for sensor in sensors:
            mask = particular_experiment["sensors"]==sensor
            particular_experiment.loc[mask,column_name] = particular_experiment.loc[mask].rolling(window_size,center=True,min_periods=1)["VOC"].mean() 
            
        timeSeriesData_BIG[outer_mask] = particular_experiment   

#create a huge dataframe to save it 

# Combine into one DataFrame, keeping the dictionary key

# Reset index so the key becomes a column
#timeSeriesData_BIG_rolling_15 = timeSeriesData_BIG.reset_index(level="seconds")

In [ ]:

window_ranges = [5,15,30,60,120]
for window_size in window_ranges:
    
    column_name = "VOC window " + str(window_size) 
    column_name_assigned = column_name + " ratio"

    timeSeriesData_BIG[column_name_assigned] = np.nan

    for index, row in userInputData.iterrows():
        column_name = "VOC window " + str(window_size)
    
    
        outer_mask = timeSeriesData_BIG["keys"] == index
        particular_experiment =  timeSeriesData_BIG.loc[outer_mask].copy()
        
        sensors = particular_experiment["sensors"].unique()
    
        for sensor in sensors:
            mask = particular_experiment["sensors"]==sensor
            ratio = (np.abs(particular_experiment.loc[mask,column_name].to_numpy()))/ (np.abs((particular_experiment.loc[mask,"VOC"].to_numpy()) + 1e-10))
            particular_experiment.loc[mask,column_name_assigned]  = ratio
        timeSeriesData_BIG.loc[outer_mask] = particular_experiment


In [ ]:
timeSeriesData_BIG

In [ ]:
def printDataBasedOnDate(column_to_print,timeSeriesData_BIG,room_other_grouping):
    
    column_names_keys_color_values = {"Id=0:BME680:breathVocEquivalent":"blue","Id=1:BME680:breathVocEquivalent":"orange","Id=2:BME680:breathVocEquivalent":"green"}
 
    
    for group_name,indexes_of_the_group in room_other_grouping.items(): 
    
        data = timeSeriesData_BIG.loc[timeSeriesData_BIG["keys"].isin(indexes_of_the_group),:]  
      # Create relplot
        g = sns.relplot(
            data=data,
            x="seconds",
            y=column_to_print,
            hue="sensors",
            col="keys",        # separate subplot per key
            kind="line",
            col_wrap=3, 
                height=7,    # default = 5
            aspect=1, # width = height × aspect (so 6 × 1.5 = 9 inches wide per subplot
            palette=column_names_keys_color_values,  # <<< ensures the same colors across all subplots  
            linewidth=2,
           facet_kws={
            "sharex": False,
            "sharey": False       
    
        })
        if ("gradient" not in column_to_print) and ("ratio" not in column_to_print):          
            #g.fig.set_size_inches(20, 16)  # width, height for total figure
            # Get the horizontal and  vertical line position for this experiment
            for key_value, ax in g.axes_dict.items():
                for column_name,color in column_names_keys_color_values.items():
        
                # Example: horizontal line from x=100 to x=500
                    ax.hlines(
                        y=userInputData.at[key_value,column_name+" before insertion median"], xmin=0, xmax=userInputData.at[key_value,"timestamp InsertingSource seconds"],
                        colors=color, linestyles="--", linewidth=1
                    )
                    ax.hlines(
                        y=userInputData.at[key_value,column_name+" after insertion median"], xmin=userInputData.at[key_value,"timestamp InsertingSource seconds"], xmax=userInputData.at[key_value,"time taken total"].total_seconds(),
                        colors=color, linestyles="--", linewidth=1
                    )
                    ax.hlines(
                        y=userInputData.at[key_value,column_name+" before insertion median"+"VOC standard scaled"], xmin=0, xmax=userInputData.at[key_value,"timestamp InsertingSource seconds"],
                        colors=color, linestyles=":", linewidth=1
                    )
                    ax.hlines(
                        y=userInputData.at[key_value,column_name+" after insertion median"+"VOC standard scaled"], xmin=userInputData.at[key_value,"timestamp InsertingSource seconds"], xmax=userInputData.at[key_value,"time taken total"].total_seconds(),
                        colors=color, linestyles=":", linewidth=1
                    )
        
                
    # Get the horizontal and  vertical line position for this experiment
        for key_value, ax in g.axes_dict.items():
            for column_name,color in column_names_keys_color_values.items():
                #value to show the time that source is inserted
                vline_x = userInputData.at[key_value, "timestamp InsertingSource seconds"]
                
                ax.axvline(x=vline_x, color="red", linestyle="--", linewidth=2)
                #value to show median in not log scaled and scaled for each sensor
                if ("gradient" not in column_to_print) and ("ratio" not in column_to_print):
                 
                    
                    name=column_name+" before insertion median"
            
                    ax.hlines(
                        y=userInputData.at[key_value,name], xmin=0, xmax=userInputData.at[key_value,"timestamp InsertingSource timedelta"].total_seconds(),
                        colors=color, linestyles="--", linewidth=2
                    )
                    name =column_name+" after insertion median"
                    
                    ax.hlines(
                        y=userInputData.at[key_value,name], xmin=userInputData.at[key_value,"timestamp InsertingSource timedelta"].total_seconds(), xmax=userInputData.at[key_value,"time taken total"].total_seconds(),
                        colors=color, linestyles="--", linewidth=1
                    )
                    
                    name =column_name+" before insertion median"+"VOC standard scaled"
                    
                    ax.hlines(
                        y=userInputData.at[key_value,name], xmin=0, xmax=userInputData.at[key_value,"timestamp InsertingSource timedelta"].total_seconds(),
                        colors=color, linestyles=":", linewidth=2
                    )
                    name =column_name+" after insertion median"+"VOC standard scaled"
                    ax.hlines(
                        y=userInputData.at[key_value,name], xmin=userInputData.at[key_value,"timestamp InsertingSource timedelta"].total_seconds(), xmax=userInputData.at[key_value,"time taken total"].total_seconds(),
                        colors=color, linestyles=":", linewidth=1
                    )
                userInputDataRow = userInputData.loc[key_value,:]
                position = f"front wall:{userInputDataRow['front-wall']},left wall:{userInputDataRow['side-left-wall']},\nright wall:{userInputDataRow['side-right-wall']},back wall:{userInputDataRow['back-wall']} \n"
                condition = f" are-doors-opened:{userInputDataRow['are-doors-opened']},are-people-inside:{userInputDataRow['are-people-inside']},\n"
                condition = condition  + f"are-windows-opened:{userInputDataRow['are-windows-opened']},are-fans-on:{userInputDataRow['are-fans-on']},\n notes:{userInputDataRow['notes']}\n"
                subtitle= f" experimentState:{userInputDataRow['experimentState']}, position: {position} , condition {condition}"
                ax.set_title(subtitle, fontsize=9)   
            g.fig.suptitle(f"Group: {group_name}", fontsize=16)
        
            g.fig.subplots_adjust(
                    top=0.75,   # space for overall title
                    wspace=0.2, # horizontal space between subplots
                    hspace=0.3 # vertical space between subplots
                   )
    
             # Overall title for the figure
        # Titles for each subplot are automatically the 'keys', but you can customize
            
                
    
        #     ax.text(
        #     0.5, 1.05, subtitle,          # x=0.5 center, y=1.05 slightly above plot
        #     transform=ax.transAxes,        # coordinates relative to axes
        #     ha='center',                   # center horizontally
        #     fontsize=10
        # )
             

In [ ]:
room_other_grouping = userInputData.groupby(["date of experiment","room"]).groups


In [ ]:
printDataBasedOnDate("VOC",timeSeriesData_BIG,room_other_grouping)

In [ ]:
printDataBasedOnDate("VOC window 30",timeSeriesData_BIG,room_other_grouping)

In [ ]:
printDataBasedOnDate("VOC rolling average standard scaled",timeSeriesData_BIG,room_other_grouping)

In [ ]:
printDataBasedOnDate("VOC gradient",timeSeriesData_BIG,room_other_grouping)

In [ ]:
printDataBasedOnDate("VOC gradient cumsum",timeSeriesData_BIG,room_other_grouping)

In [ ]:
timeSeriesData_BIG

In [ ]:
grad_ratio_df = pd.DataFrame(index=range(userInputData.shape[0])) 
window_ranges = [5,15,30,60,120]
for window_size in window_ranges:
    
    column_name = "VOC window " + str(window_size)
    timeSeriesData_BIG

    grad_ratio_df[column_name] = np.nan
        
for window_size in window_ranges:
    column_name = "VOC window " + str(window_size)

    for index, row in userInputData.iterrows():
            outer_mask = timeSeriesData_BIG["keys"] == index
            particular_experiment =  timeSeriesData_BIG.loc[outer_mask].copy()
            
            sensors = particular_experiment["sensors"].unique()
            mean =0
            for sensor in sensors:
                column_name_par = column_name + sensor
                mask = particular_experiment["sensors"]==sensor
                grad_ratio = np.abs(particular_experiment.loc[mask,column_name].to_numpy()) / np.abs((particular_experiment.loc[mask,"VOC"].to_numpy()) + 1e-10)
                mean = mean + grad_ratio.mean()
            grad_ratio_df.loc[index,column_name]  =  mean/sensors.shape[0]
        

# Relative change in gradient magnitude


In [ ]:
grad_ratio_df.tail(60)